## CS313 - Chapter 7 Homework

## Medium Article 1: _Mastering SQL Window Functions: A Comprehensive Tutorial_ (2023) by Manu Mulaveesala

## Medium Article 2:  _PIVOT And UNPIVOT for Data Analysis in SQL Server_ (2024) by Gianpero Andrenacci

## Proposition 1: Show the running Total per customer[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-1:-Show-the-running-Total-per-customer)

### Functional Specification: Calculate cumulative quantity by customer, showing running total for every order. Use a window function with partition by CustomerID, Order by OrderDate and OrderID.[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Functional-Specification:-Calculate-cumulative-quantity-by-customer,-showing-running-total-for-every-order.-Use-a-window-function-with-partition-by-CustomerID,--Order-by-OrderDate-and-OrderID.)

```
Medium Article: This query exercises the running total concept that is dicussed in Manu Mulaveesala's article "Mastering SQL Window Functions: A Comprehensive Tutorial" (Medium, 2023)."If you want to count how many blocks TOTAL you have in a column or add up their numbers, a Window Function can do that for you, looking at each block one by one and keeping a running total."
```

#### [](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Medium-Article:-This-query-exercises-the-running-total-concept-that-is-dicussed-in-Manu-Mulaveesala's-article-%22Mastering-SQL-Window-Functions:-A-Comprehensive-Tutorial%22-(Medium,-2023).)

In [6]:
SELECT
    c.CustomerID,
    o.OrderID,
    od.Quantity,
    SUM(od.Quantity) OVER (
        PARTITION BY c.CustomerID
        ORDER BY o.OrderDate, o.OrderID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS RunningTotal
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o ON c.CustomerID = o.CustomerID
JOIN Sales.OrderDetail AS od ON o.OrderID = od.OrderID;


(2155 rows affected)

Total execution time: 00:00:00.065

CustomerID,OrderID,Quantity,RunningTotal
1,10643,15,15
1,10643,21,36
1,10643,2,38
1,10692,20,58
1,10702,6,64
1,10702,15,79
1,10835,15,94
1,10835,2,96
1,10952,16,112
1,10952,2,114


## Proposition 2: Rank customer by total quantity ordered[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-2:-Rank-customer-by-total-quantity-ordered)

### Functional Specification: Calculate total quantity per customer and assign ranks with Rank(), Dense\_Rank() for no rank gaps, Row\_Number() for no rank duplicates

In [7]:
SELECT
    o.CustomerID,
    SUM(od.Quantity) AS TotalQuantity,
    RANK() OVER (
        ORDER BY SUM(od.Quantity) DESC
    ) AS CustomerRank,
    DENSE_RANK() OVER (
        ORDER BY SUM(od.Quantity) DESC
    ) AS DenseCustomerRank,
    ROW_NUMBER() OVER (
        ORDER BY SUM(od.Quantity) DESC
    ) AS RowNum
FROM Sales.OrderDetail AS od
JOIN Sales.[Order] AS o ON od.OrderID = o.OrderID
GROUP BY o.CustomerID
ORDER BY CustomerRank;


(89 rows affected)

Total execution time: 00:00:00.055

CustomerID,TotalQuantity,CustomerRank,DenseCustomerRank,RowNum
71,4958,1,1,1
20,4543,2,2,2
63,3961,3,3,3
37,1684,4,4,4
25,1525,5,5,5
65,1383,6,6,6
24,1234,7,7,7
35,1096,8,8,8
76,1072,9,9,9
89,1063,10,10,10


## Proposition 3: Show each customer's order quantity along with next order quantities[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-3:-Show-each-customer's-order-quantity-along-with-next-order-quantities)

### Functional Specification: Partition by CustomerID and Order by Orderdate and OrderID, use LEAD to return value of next row in the window

In [8]:
SELECT
    o.CustomerID,
    o.OrderID,
    o.OrderDate,
    od.Quantity,
    LEAD(od.Quantity) OVER (
        PARTITION BY o.CustomerID
        ORDER BY o.OrderDate, o.OrderID
    ) AS NextQuantity
FROM Sales.OrderDetail AS od
JOIN Sales.[Order] AS o ON od.OrderID = o.OrderID
ORDER BY o.CustomerID, o.OrderDate

(2155 rows affected)

Total execution time: 00:00:00.142

CustomerID,OrderID,OrderDate,Quantity,NextQuantity
1,10643,2015-08-25,15,21
1,10643,2015-08-25,21,2
1,10643,2015-08-25,2,20
1,10692,2015-10-03,20,6
1,10702,2015-10-13,6,15
1,10702,2015-10-13,15,15
1,10835,2016-01-15,15,2
1,10835,2016-01-15,2,16
1,10952,2016-03-16,16,2
1,10952,2016-03-16,2,40


## Proposition 4: Show each product's order quantity as a percent of total quantity ordered and as a percent of all quantities of that product[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-4:-Show-each-product's-order-quantity-as-a-percent-of-total-quantity-ordered-and-as-a-percent-of-all-quantities-of-that-product)

### Functional Specification: Use Sum() OVER() to get a percentage out of all, and OVER(PARTITION BY ProductID) to compute over specific product

In [9]:
SELECT 
    od.OrderID,
    od.ProductID,
    od.Quantity,
    100.0 * od.Quantity / 
        SUM(od.Quantity) OVER() AS PercentOutAll,
    100.0 * od.Quantity / 
        SUM(od.Quantity) OVER(PARTITION BY od.ProductID) AS PercenOutProduct
FROM Sales.OrderDetail AS od
ORDER BY od.ProductID, od.OrderID;


(2155 rows affected)

Total execution time: 00:00:00.143

OrderID,ProductID,Quantity,PercenOutAll,PercenOutProduct
10285,1,45,0.087690239102,5.434782608695
10294,1,18,0.035076095640,2.173913043478
10317,1,20,0.038973439600,2.415458937198
10348,1,15,0.029230079700,1.811594202898
10354,1,12,0.023384063760,1.449275362318
10370,1,15,0.029230079700,1.811594202898
10406,1,10,0.019486719800,1.207729468599
10413,1,24,0.046768127521,2.898550724637
10477,1,15,0.029230079700,1.811594202898
10522,1,40,0.077946879201,4.830917874396


## Propostion 5: Count orders per employees selected (71, 20, 63, 24,37)[¶](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Propostion-5:-Count-orders-per-employees-selected-(71,-20,-63,-24,37))

### Functional Specification: Transform order info from rows to columns to show how many orders were placed by each customers, for each employee. Use PIVOT to aggregate order counts by CustomerID

Medium Article: This article applies the concepts from the article _PIVOT And UNPIVOT for Data Analysis in SQL Server_ <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">(2024) by Gianpero Andrenacci. The article demonstrates how PIVOT converts row values into separate columns using an aggregate function. For example, this query pivors CustomerID values 71,20,63,24,37 into their own separate columns, allowing for easier comparisons between them versus the original data orientation.</span>

In [11]:
SELECT 
    EmployeeID,
    [71] AS Cust71,
    [20] AS Cust20,
    [63] AS Cust63,
    [24] AS Cust24,
    [37] AS Cust37
FROM (
    SELECT
        EmployeeID,
        CustomerID,
        OrderID
    FROM Sales.[Order]
) AS SUB
PIVOT (
    COUNT(OrderID)
    FOR CustomerID IN ([71], [20], [63], [24], [37])
) AS p
ORDER BY EmployeeID;


(9 rows affected)

Total execution time: 00:00:00.169

EmployeeID,Cust71,Cust20,Cust63,Cust24,Cust37
1,6,5,4,1,1
2,4,3,6,3,3
3,2,4,5,2,5
4,4,5,5,2,1
5,3,0,2,1,0
6,4,2,0,2,3
7,3,4,1,2,2
8,4,4,4,6,1
9,1,3,1,0,3


## Proposition 6: Unpivot order information back into rows from columns to show order counts for each employee-customer combination.[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-6:-Unpivot-order-information-back-into-rows-from-columns-to-show-order-counts-for-each-employee-customer-combination.)

### Functional Specification: Unpivot the order information using CROSS APPLY, where OrderCount \> 0 to filter out rows with no orders[¶](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Functional-Specification:-Unpivot-the-order-information-using-CROSS-APPLY,-where-OrderCount-%3E-0-to-filter-out-rows-with-no-orders)

In [12]:
SELECT 
    EmployeeID,
    Customerid,
    Ordercount
FROM (
    SELECT 
        EmployeeID,
        [71] AS Cust71,
        [20] AS Cust20,
        [63] AS Cust63,
        [24] AS Cust24,
        [37] AS Cust37
    FROM (
        SELECT
            EmployeeID,
            CustomerID,
            OrderID
        FROM Sales.[Order]
    ) AS src
    PIVOT (
        COUNT(OrderID)
        FOR CustomerID IN ([71], [20], [63], [24], [37])
    ) AS p
) AS pivoted
CROSS APPLY (VALUES
    ('71', Cust71),
    ('20', Cust20),
    ('63', Cust63),
    ('24', Cust24),
    ('37', Cust37)
) AS Unpivoted(CustomerId, OrderCount)
WHERE ordercount > 0
ORDER BY EmployeeID, CustomerId;


(41 rows affected)

Total execution time: 00:00:00.497

EmployeeID,Customerid,Ordercount
1,20,5
1,24,1
1,37,1
1,63,4
1,71,6
2,20,3
2,24,3
2,37,3
2,63,6
2,71,4


## Proposition 7: Calculate the total quantities by each employee and customer combination, per employee across all possible customers, per customer and the total across all employees and customers.[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-7:-Calculate-the-total-quantities-by-each-employee-and-customer-combination,-per-employee-across-all-possible-customers,-per-customer-and-the-total-across-all-employees-and-customers.)

### Functional Specification: Calculate orders quantities across each specified level/set using GROUPING SETS (per employee/customer combination, per employee, per customer, and total across all

In [13]:
SELECT 
    o.EmployeeID,
    o.CustomerID,
    SUM(od.Quantity) as TotalQuantity
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od
    ON o.OrderID = od.OrderID
GROUP BY GROUPING SETS
(
    (o.EmployeeID, o.CustomerID),  
    (o.EmployeeID),               
    (o.CustomerID),                
    ()                             
)
ORDER BY o.EmployeeID, o.CustomerID;


(563 rows affected)

Total execution time: 00:00:00.073

EmployeeID,CustomerID,TotalQuantity
NULL,NULL,51317
NULL,1,174
NULL,2,63
NULL,3,359
NULL,4,650
NULL,5,1001
NULL,6,140
NULL,7,666
NULL,8,190
NULL,9,980


## Proposition 8: Calculate total quantity of orders, by year, month and day, showing quantity per day and total quantityy across all dates.[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-8:-Calculate-total-quantity-of-orders,-by-year,-month-and-day,-showing-quantity-per-day-and-total-quantityy-across-all-dates.)

### Functional Specification: Use ROLLUP to calculate order quantities at each date (year, month, day). Return grand total, total per year, month and day

In [18]:
SELECT 
    YEAR(o.OrderDate) AS OrderYear,
    MONTH(o.OrderDate) AS OrderMonth,
    DAY(o.OrderDate) AS OrderDay,
    SUM(od.Quantity) AS TotalQuantity
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od ON o.OrderID = od.OrderID
GROUP BY ROLLUP(YEAR(o.OrderDate), MONTH(o.OrderDate), DAY(o.OrderDate))
ORDER BY OrderYear, OrderMonth, OrderDay;


(507 rows affected)

Total execution time: 00:00:00.053

OrderYear,OrderMonth,OrderDay,TotalQuantity
NULL,NULL,NULL,51317
2014,NULL,NULL,9581
2014,7,NULL,1462
2014,7,4,27
2014,7,5,49
2014,7,8,101
2014,7,9,105
2014,7,10,102
2014,7,11,57
2014,7,12,110


## Proposition 9: Show every combination of employee and customer, including subtotals for each employee across all customer, each customer across all employees as well as the total across all combinations.[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-9:-Show-every-combination-of-employee-and-customer,-including-subtotals-for-each-employee-across-all-employees,-each-customer-across-all-employees-as-well-as-the-total-across-all-combinations.)

### Functional Specification: Apply CUBE operator to get all combinations of employee/customer, per employee, per customer and total across all combinations.

In [24]:
SELECT 
    o.EmployeeID, 
    c.CustomerId, 
    SUM(od.Quantity) AS SumQuantity
FROM Sales.[Order] AS o
JOIN Sales.Customer AS c ON o.CustomerID = c.CustomerID
JOIN Sales.OrderDetail AS od ON o.OrderID = od.OrderID
GROUP BY CUBE(o.EmployeeID, c.CustomerID)
ORDER BY o.EmployeeID, c.CustomerId


(563 rows affected)

Total execution time: 00:00:00.040

EmployeeID,CustomerId,SumQuantity
NULL,NULL,51317
NULL,1,174
NULL,2,63
NULL,3,359
NULL,4,650
NULL,5,1001
NULL,6,140
NULL,7,666
NULL,8,190
NULL,9,980


## Proposition 10: Show every combination of employee and customer, including subtotals for each employee across all employees, each customer across all employees as well as the total across all combinations, and assign a group number for each employeeID where null is a different group[](https://jupyter.org/try-jupyter/notebooks/?path=Individual_HwGroup5_AlexanderBulatao.ipynb#Proposition-10:-Show-every-combination-of-employee-and-customer,-including-subtotals-for-each-employee-across-all-employees,-each-customer-across-all-employees-as-well-as-the-total-across-all-combinations,-and-assign-a-group-number-for-each-employeeID.)

### Functional Specification: Apply CUBE operator to get employee/customer combination, per employee, per customer and total across all combinations. Assign a groupID using GROUPING\_ID

In [25]:
SELECT
    o.EmployeeID,
    o.CustomerID,
    SUM(od.Quantity) AS TotalQuantity,
    GROUPING_ID(o.EmployeeID) AS GroupID
FROM Sales.[Order] AS o
JOIN Sales.OrderDetail AS od
    ON o.OrderID = od.OrderID
GROUP BY CUBE(o.EmployeeID, o.CustomerID)
ORDER BY o.EmployeeID, o.CustomerID;


(563 rows affected)

Total execution time: 00:00:00.095

EmployeeID,CustomerID,TotalQuantity,GroupID
NULL,NULL,51317,1
NULL,1,174,1
NULL,2,63,1
NULL,3,359,1
NULL,4,650,1
NULL,5,1001,1
NULL,6,140,1
NULL,7,666,1
NULL,8,190,1
NULL,9,980,1
